# 03 — Convergence Metrics

For the transport-only step we evaluate one optimal plan returned by POT
and the realized quantities

$$\hat\alpha=-\mathfrak D/W_2^2,\quad
  \hat L=\|V\|/W_2,\quad
  \beta_\star=-\mathfrak D/\|V\|^2.$$

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from msflow import *
import msflow

pd.set_option("display.width", 180)

## 1 · 1D global counterexample

For the Gaussian population row field, the configuration below gives
$\mathfrak D>0$.  The value $R>\sigma\sqrt{\log15}$ is a **sufficient
threshold for the lower bound** $\mathfrak D\ge R^2/8$; it is not the exact
zero-crossing of $\mathfrak D$.  This is strong negative evidence for any
global positive dissipativity or star-cocoercivity claim.

In [ ]:
fig = plot_two_dirac_counterexample(R=3.0, sigma=1.0)
save(fig, "03_counterexample");

In [ ]:
rows = [{"R/sigma": R, **{k: v for k, v in two_dirac_dissipativity(R, 1.0).items()
                          if k in ("D", "bound", "V_half", "V_three_quarter")}}
        for R in [0.5, 1.0, 1.5, 1.6456, 2.0, 3.0, 4.0]]
ce = pd.DataFrame(rows).set_index("R/sigma")
ce["dissipative?"] = np.where(ce["D"] < 0, "yes", "NO")
save_table(ce, "03_counterexample",
           caption=r"Two-Dirac counterexample: the sufficient lower-bound regime begins at "
                   r"$R/\sigma=\sqrt{\log 15}\approx1.646$.")
ce.round(4)

---
## 2 · Frozen bandwidth sweep

$\beta_\star$ changes under a scalar rescaling of the field and may grow
when $\|V\|$ collapses.  We therefore display it together with field norm,
scale-free OT alignment, $\hat\alpha$ and $\hat L$.  The curves describe two
frozen states; their shape is an observation, not an assumed annealing law.

In [ ]:
data = load_dataset("ring", n=320, seed=2026)
TAU, CLIP = 0.04, None
PROBE_EVERY, STORE_EVERY = 25, 50
SIGMAS = np.logspace(-1, 1.0, 26)
rng = np.random.default_rng(3)

states = {
    "far ($q^0$)": data.q0,
    "close to $p$": Emp(data.p.X[rng.choice(data.p.n, 320)]
                         + 0.4 * rng.standard_normal((320, 2))),
}
curves, diagnostics = {}, []
for sname, q in states.items():
    for kname in ("Gaussian", "Laplacian"):
        beta = beta_vs_sigma(data.p, q, SIGMAS,
                             lambda s, kn=kname: mean_shift(get_kernel(kn, s)))
        curves[f"{kname}, {sname}"] = beta
        for s, b in zip(SIGMAS, beta):
            V = mean_shift(get_kernel(kname, float(s)))(data.p, q)
            d = step_diagnostics(data.p, q, V, TAU)
            diagnostics.append({"state": sname, "kernel": kname, "sigma": s,
                                "beta_star": b, "Vnorm": d["Vnorm"],
                                "ot_alignment": d["ot_alignment"],
                                "alpha_hat": d["alpha_hat"], "L_hat": d["L_hat"]})

fig = plot_beta_vs_sigma(SIGMAS, curves, tau=TAU,
                         title="realized star-cocoercivity at frozen states")
save(fig, "03_beta_vs_sigma")
frozen = pd.DataFrame(diagnostics)
fig, ax = plt.subplots(1, 3, figsize=(9.6, 3.2))
for (state, kernel), group in frozen.groupby(["state", "kernel"]):
    label = f"{kernel}, {state}"
    ax[0].loglog(group.sigma, group.Vnorm, label=label)
    ax[1].semilogx(group.sigma, group.ot_alignment)
    ax[2].loglog(group.sigma, group.L_hat)
ax[0].set(xlabel=r"$\sigma$", ylabel=r"$\|V\|$", title="field magnitude")
ax[1].set(xlabel=r"$\sigma$", ylabel="OT alignment", title="scale-free direction")
ax[2].set(xlabel=r"$\sigma$", ylabel=r"$\hat L$", title="relative magnitude")
ax[0].legend(fontsize=6)
for a in ax: a.grid(alpha=.3, which="both")
fig.tight_layout(); save(fig, "03_frozen_scale_diagnostics")

In [ ]:
SIGMA, N_ITER = 1.0, 600

DENG_SELF_MASK = True
DENG_SELF_MASK_DISTANCE = 1e6

SINKHORN_TOL = 1e-8
SINKHORN_MAX_ITER = 4000

kern = get_kernel("Laplacian", SIGMA)

row_field = deng_population(kern)

fields = {
    "ot": ot(),

    "deng_theoretical_row": row_field,

    "deng_official": deng_official(
        temperature=SIGMA,
        self_mask=DENG_SELF_MASK,
        self_mask_distance=DENG_SELF_MASK_DISTANCE,
    ),

    "sinkhorn_proxy": sinkhorn_proxy(
        get_kernel("Gaussian", SIGMA)
    ),

    "sharp_core_nm": norm_matched(
        sharp_core(kern),
        row_field,
    ),
    "ms_unnorm": mean_shift_unnormalized(kern),
    "ms_unnorm_nm": norm_matched(
        mean_shift_unnormalized(kern),
        row_field,
    ),

    "sinkhorn_K5": sinkhorn_drift(
        SIGMA,
        K=5,
    ),

    "sinkhorn_Kinf": sinkhorn_drift(
        SIGMA,
        K=np.inf,
        tol=SINKHORN_TOL,
        max_iter=SINKHORN_MAX_ITER,
    ),
}

cert_runs = Runs()

for name, fld in fields.items():
    probes = dict(certificate_probes(TAU))

    probes["Fcoco"] = (
        lambda k, p, q, V:
        f_dissipativity(p, q, kern, V)["beta_F"]
    )

    cert_runs[name] = run(
        data.q0,
        data.p,
        fld,
        n_iter=N_ITER,
        tau=TAU,
        clip=CLIP,
        probes=probes,
        probe_every=PROBE_EVERY,
        store_every=STORE_EVERY,
        label=name,
    )

fig = plot_certificates(cert_runs, tau=TAU)
save(fig, "03_certificates")

In [ ]:
def cert_summary(r):
    b = r.probes["beta_star"]
    return {"sampled min beta_star": float(np.nanmin(b)),
            "sampled median beta_star": float(np.nanmedian(b)),
            "% probed KM ok": pct_above(b, TAU / 2),
            "% probed rho_diss < 0": pct_below(r.probes["rho_diss"], 0.0),
            "sampled max rho_NE": float(np.nanmax(r.probes["rho_NE"])),
            "median beta_F (Laplacian F)": float(np.nanmedian(r.probes["Fcoco"]))}

cert_table = runs_table(cert_runs, columns=("W2",), extra=cert_summary)
save_table(
    cert_table, "03_certificates",
    caption=("Selected-plan diagnostics on probed ring states "
             f"(sigma={SIGMA:g}, tau={TAU:g}, stride={PROBE_EVERY})."))
cert_table.round(4)

## 3 · Realized $(\mathcal H)$ ratios on the probed states

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(9.6, 3.2))
for name, r in cert_runs.items():
    c = color_of(name)
    ax[0].plot(r.probe_k, r.probes["alpha_hat"], color=c, lw=1.7, label=r.label)
    ax[1].plot(r.probe_k, r.probes["L_hat"], color=c, lw=1.7)
    ax[2].plot(r.probe_k, r.probes["tau_opt"], color=c, lw=1.7)
ax[0].axhline(0, color="k", ls=":", lw=1)
ax[0].set(xlabel="$k$", ylabel=r"$\hat\alpha$",
          title=r"realized dissipativity ratio")
ax[0].legend(fontsize=7)
ax[1].axhline(1, color="0.5", ls="--", lw=1, label="map-OT value 1")
ax[1].set(xlabel="$k$", ylabel=r"$\hat L$",
          title=r"realized relative field norm")
ax[1].legend(fontsize=7)
ax[2].axhline(TAU, color="#d62728", ls="--", lw=1.4,
              label=rf"$\tau={TAU}$ in use")
ax[2].set(xlabel="$k$", ylabel=r"$\hat\alpha/\hat L^2$", yscale="symlog",
          title=r"pointwise quadratic-bound optimum")
ax[2].legend(fontsize=7)
for a in ax: a.grid(alpha=.3)
fig.tight_layout(); save(fig, "03_H_constants")

In [ ]:
def H_summary(r):
    a, L = r.probes["alpha_hat"], r.probes["L_hat"]
    sampled_min_a, sampled_max_L = float(np.nanmin(a)), float(np.nanmax(L))
    tau_max = (2 * sampled_min_a / sampled_max_L ** 2
               if sampled_max_L > 0 else np.nan)
    return {"sampled min alpha_hat": sampled_min_a,
            "sampled max L_hat": sampled_max_L,
            "decoupled beta summary": (sampled_min_a / sampled_max_L ** 2
                                         if sampled_max_L > 0 else np.nan),
            "sampled-summary tau max": tau_max,
            "tau used": r.tau,
            "sampled-summary headroom": (tau_max / r.tau if tau_max > 0 else np.nan),
            "median pointwise tau_opt": float(np.nanmedian(r.probes["tau_opt"])),
            "sampled summary usable?": "yes" if sampled_min_a > 0 and r.tau < tau_max else "no",
            "decoupled rho bound": float(np.sqrt(max(
                1 - 2 * sampled_min_a * r.tau + (sampled_max_L * r.tau) ** 2, 0.0)))}

H_table = runs_table(cert_runs, columns=("W2",), extra=H_summary)
save_table(H_table, "03_H_constants",
           caption="Minima/maxima over probed states; not uniform constants on a ball.")
H_table.round(4)

---
## 4 · $\mathcal F$ alignment

In [ ]:
fdiss_runs = Runs()
for name in ("deng_theoretical_row", "sharp_core_nm", "ms_unnorm"):
    fdiss_runs[name] = run(
        data.q0, data.p, fields[name], n_iter=N_ITER, tau=TAU, clip=CLIP,
        probes=standard_probes(data, kern), probe_every=20,
        store_every=100, label=name)
fig = plot_f_dissipativity(fdiss_runs, tau=TAU)
save(fig, "03_f_dissipativity")

fd = runs_table(fdiss_runs, columns=("W2", "F", "beta_F"),
                extra=lambda r: {
                    "median |mechant|/|quad|": float(np.nanmedian(
                        np.abs(r.probes["mechant"]) / (np.abs(r.probes["quad"]) + 1e-14))),
                    "% probed mechant < 0": pct_below(r.probes["mechant"], 0.0),
                    "% probed beta_F < tau/2 (reference only)":
                        pct_below(r.probes["beta_F"], TAU / 2)})
save_table(fd, "03_f_dissipativity",
           caption="F-alignment decomposition on sampled trajectory states.")
fd.round(4)

---
## 5 · Annealing

In [ ]:
S0, SINF, POLY_POWER = 3.0, 0.3, 2.0
schedules = {
    "fixed_1.0": constant(1.0),
    "fixed_0.3": constant(SINF),
    "exp_fast": exp_anneal(S0, SINF, N_ITER / 8),
    "exp_slow": exp_anneal(S0, SINF, N_ITER / 2),
    "linear": linear_anneal(S0, SINF, N_ITER),
    "cosine": cosine_anneal(S0, SINF, N_ITER),
    "polynomial": poly_anneal(S0, SINF, N_ITER, power=POLY_POWER),
    "plateau_exp": plateau_then_exp(S0, SINF, N_ITER, 0.4, 0.15),
}
sched_runs = Runs()
for name, sched in schedules.items():
    probes = dict(certificate_probes(TAU))
    probes["n_covered"] = lambda k, p, q, V: mode_metrics(q, data.centers)["n_covered"]
    probes["H"] = lambda k, p, q, V: smoothed_kl(q, p, 0.5, data.extent)
    sched_runs[name] = run(
        data.q0, data.p, lambda s: mean_shift(get_kernel("Laplacian", s)),
        schedule=sched, n_iter=N_ITER, tau=TAU, clip=CLIP,
        probes=probes, probe_every=PROBE_EVERY, store_every=STORE_EVERY,
        label=name)

fig = plot_schedules(sched_runs); save(fig, "03_schedules")
fig = plot_runs(sched_runs, keys=("W2", "H", "n_covered", "beta_star"),
                hlines={"beta_star": TAU / 2})
save(fig, "03_schedule_curves")

In [ ]:
sched_table = runs_table(sched_runs, columns=("W2", "H", "n_covered"),
                         n_modes=len(data.centers), tau=TAU,
                         extra=lambda r: {"min beta_star": float(np.nanmin(r.probes["beta_star"])),
                                          "max rho_NE": float(np.nanmax(r.probes["rho_NE"]))})
save_table(sched_table, "03_schedules",
           caption=r"Bandwidth schedules: speed versus the convergence certificates.")
sched_table.round(4)

## 6 · Diffusion Regularized Drifting

Here `score_p` is the Gaussian fitted to the empirical target moments while
$W_2$ is computed to the finite target cloud, and `score_q` is a KDE.

In [ ]:
SCORE_N, SCORE_SEED = 400, 2026
SCORE_KERNEL_SIGMA, SCORE_EVAL_SIGMA = 1.0, 0.5
SCORE_BETAS = [0.0, 0.1, 0.3, 1.0]
SCORE_TAU, SCORE_N_ITER = 0.02, 800
SCORE_PROBE_EVERY, SCORE_STORE_EVERY = 25, 100
SCORE_CLIP = None

gauss = load_dataset("gaussian_aniso", n=SCORE_N, seed=SCORE_SEED)
Sig_p, mu_p = gauss.p.cov(), gauss.p.mean()
kappa = 1.0 / np.linalg.eigvalsh(Sig_p).max()
print(f"target eigenvalues {np.round(np.linalg.eigvalsh(Sig_p), 3)}  ->  kappa = {kappa:.3f}")

kern_l = get_kernel("Laplacian", SCORE_KERNEL_SIGMA)
base = mean_shift(kern_l)
score_p = gaussian_score(mu_p, Sig_p)

beta_runs = Runs()
for beta in SCORE_BETAS:
    fld = base if beta == 0 else score_regularized(base, beta, kern_l, score_p=score_p)
    probes = dict(certificate_probes(SCORE_TAU))
    probes["H"] = lambda k, p, q, V: smoothed_kl(
        q, p, SCORE_EVAL_SIGMA, gauss.extent)
    beta_runs[f"beta{beta}"] = run(
        gauss.q0, gauss.p, fld, n_iter=SCORE_N_ITER, tau=SCORE_TAU,
        clip=SCORE_CLIP, probes=probes, probe_every=SCORE_PROBE_EVERY,
        store_every=SCORE_STORE_EVERY, label=rf"$\beta={beta}$")

fig = plot_runs(
    beta_runs, keys=("W2", "H", "beta_star", "L_hat", "ot_alignment", "rho_diss"),
    ncol=3, figsize=(9.0, 6.2),
    hlines={"beta_star": SCORE_TAU / 2, "rho_diss": 0.0})
save(fig, "03_diffusion_regularizer")

In [ ]:
diff_table = runs_table(
    beta_runs, columns=("W2", "H"), tau=SCORE_TAU,
    extra=lambda r: {
        "sampled min beta_star": float(np.nanmin(r.probes["beta_star"])),
        "sampled max L_hat": float(np.nanmax(r.probes["L_hat"])),
        "median OT alignment": float(np.nanmedian(r.probes["ot_alignment"])),
        "% probed rho_diss < 0": pct_below(r.probes["rho_diss"], 0.0)})
save_table(
    diff_table, "03_diffusion_regularizer",
    caption=("Score-plugin ablation: analytic Gaussian score fitted to the "
             "sampled target moments, KDE model score, empirical-cloud W2."))
diff_table.round(4)

### Visualizing Diffusion Drifting

In [ ]:
SCORE_ANIM_BETAS = (0.0, 1)
SCORE_ANIM_EVERY_ITER = 20
SCORE_ANIM_INTERVAL_MS = 80
SCORE_ANIM_PANEL_SIZE = 2.8
SCORE_ANIM_MAX_WIDTH = 620

animation_runs = Runs()
for beta in SCORE_ANIM_BETAS:
    key = f"beta{float(beta)}"
    if key not in beta_runs:
        raise ValueError(
            f"beta={beta:g} was not simulated; choose values from SCORE_BETAS={SCORE_BETAS}"
        )
    animation_runs[key] = beta_runs[key]

diffusion_animation = animate_particles(
    animation_runs, gauss.p, extent=gauss.extent,
    every_iter=SCORE_ANIM_EVERY_ITER,
    interval=SCORE_ANIM_INTERVAL_MS,
    panel_size=SCORE_ANIM_PANEL_SIZE,
)
animation_html(diffusion_animation, max_width=SCORE_ANIM_MAX_WIDTH)